# Stamm SDK -- the complete guide, start to finish

This is the one notebook to read if you've never used `stamm-sdk` before. It walks through every stage of the pipeline in order, explaining *why* each step exists before showing the code for it:

```
install -> login -> discover models -> inspect a model's schema
   -> prepare your data (map columns, validate, pad with synthetic rows if needed)
   -> Gate 1: validate the raw data
   -> train (two ways: SDK-managed, or bring-your-own-training)
   -> Gate 2: validate the trained result, with a deliberate bad actor to prove the defense works
   -> aggregate into a new global model
   -> persist the round back to the registry
   -> (bonus) predict() for models you can't train locally
```

Companion documents, for when you want the full reference instead of a guided walkthrough:
- `docs/SDK_GUIDE.md` -- every module, every import, in table form.
- `docs/MODEL_CATALOG.md` -- which specific models in this registry actually work for training today, and with which hyperparameters.

Everything in this notebook runs against the real, live registry -- no mocking. The "attacker" and "synthetic data" steps are deliberately adversarial/fake *inputs*, not a fake *server*.

## 0. Install

Base install covers auth and read-only registry discovery. Training needs the `[learning]` extra (adds `scikit-learn`, `torch`, `joblib`) -- without it, `load_model()`/`train()` raise a clear message telling you to install it, rather than a confusing `ImportError`.

In [ ]:
!pip -q install --upgrade "stamm-sdk[learning]"

## 1. Authenticate

Everything starts from a `Client`. It wraps an authenticated HTTP session; `sdk.ml` (an `APIClient`) is where every registry call actually lives.

`Client(base_url=...)` defaults to `STAMM_API_BASE_URL` if you don't pass one, and that in turn defaults to this registry's real address -- so in practice you often don't need to pass `base_url` at all.

In [ ]:
from getpass import getpass

import numpy as np
import pandas as pd

from stamm_sdk import Client

email = input("Email: ")
password = getpass("Password (hidden): ")

sdk = Client(base_url="http://188.245.241.118:8080")
sdk.login(email=email, password=password)
print("Logged in.")

## 2. Discover what you can work with

`list_available_models()` resolves the projects you belong to (the registry already filters this server-side by lab membership) and fans out across them -- you never need to know a project id in advance just to see what's there.

In [ ]:
available = sdk.ml.list_available_models()
print(f"{len(available)} models visible to you\n")
for m in available:
    print(f"  {m['project_id']:8} {m['slug']:35} algorithm={m['algorithm']}")

## 3. Inspect one model's schema in detail

`get_schemas()` is the single source of truth for "what does this model need, and how do I train it." It normalizes the registry's own metadata (which, under the hood, mixes a database row and a richer YAML description) into one typed `ModelSchema`:

- `algorithm` -- what the registry calls it (`"random_forest"`, `"svm"`, `"rnn"`, ...)
- `framework` -- what you actually need to train/load it: `"sklearn"`, `"pytorch"`, `"keras"`, or `"r"`. This is detected from the model's *real* declared packages, not guessed from `algorithm` -- a model with `algorithm="rnn"` in this registry actually turned out to be Keras, not PyTorch, and guessing wrong would silently try the wrong deserializer.
- `features` -- a list of `FeatureSpec(name, type, lag, required, range_min, range_max)`, used by everything downstream: data mapping, synthetic generation, Gate 1.
- `output_name` -- the target column name.

We'll use `0001_python_penicillin_RF` for the rest of this notebook -- it's the one model in this registry with zero rough edges (see `docs/MODEL_CATALOG.md` for the full list of what else is available and what's caveated).

In [ ]:
project_id = "P0001"
model_id = "0001_python_penicillin_RF"

schema = sdk.ml.get_schemas(project_id, model_id)

print("algorithm:  ", schema.algorithm)
print("framework:  ", schema.framework)
print("output_name:", schema.output_name)
print("features:")
for f in schema.features:
    print(f"  - {f.name:35} range=[{f.range_min}, {f.range_max}]  lag={f.lag}  required={f.required}")

## 4. Prepare your data

### 4a. Map your own column names onto the model's feature names

In practice your raw data almost never already has the exact column names the model expects. `map_input_columns` renames explicitly from a `{your_name: feature_name}` dict you provide -- it never guesses or fuzzy-matches, on purpose: a silent wrong guess here would train on the wrong column without any warning. `validate_mapped_data` then checks every *required* feature actually landed after the rename.

Below, `my_raw_data` stands in for "whatever CSV/dataframe you actually have." It's built generically from the schema here (so this cell works no matter which model you point the notebook at), with deliberately wrong column names (`raw_<feature>`) to exercise the real mapping step.

In [ ]:
from stamm_sdk.data import map_input_columns, validate_mapped_data

rng = np.random.default_rng(0)
my_raw_data = pd.DataFrame({
    f"raw_{f.name}": rng.uniform(f.range_min or 0.0, f.range_max or 1.0, size=8)
    for f in schema.features
})

column_map = {f"raw_{f.name}": f.name for f in schema.features}
mapped = map_input_columns(my_raw_data, schema, column_map)

report = validate_mapped_data(mapped, schema)
print("passed:", report.passed, "| missing:", report.missing_features, "| type mismatches:", report.type_mismatches)

In [ ]:
# And what it looks like when mapping is INCOMPLETE -- drop the last
# feature's mapping on purpose, to see validate_mapped_data catch it:
incomplete_map = dict(list(column_map.items())[:-1])
bad = map_input_columns(my_raw_data, schema, incomplete_map)
bad_report = validate_mapped_data(bad, schema)
print("passed:", bad_report.passed, "| missing:", bad_report.missing_features)

### 4b. Not enough rows? Generate synthetic data

8 rows isn't enough to train anything real. `generate_synthetic_rows` fills the gap: uniform-random within each feature's actual physical range (from the model's own metadata when available, falling back to your real data's min/max, and raising rather than guessing if neither exists). Every synthetic row is tagged `is_synthetic=True` so it stays traceable -- it's never silently mixed in as if it were real.

Since we don't have a real labeled target for either the real or synthetic rows in this demo, we fake one as a noisy linear function of the inputs purely so there's *something* to train against below -- in real use you'd already have real measured outcomes for your real rows, and only the inputs need padding out.

In [ ]:
from stamm_sdk.data import generate_synthetic_rows

TARGET_ROWS = 60
synthetic = generate_synthetic_rows(schema, mapped, target_n=TARGET_ROWS, seed=1)
print(f"had {len(mapped)} real rows, generated {len(synthetic)} synthetic rows")

combined = pd.concat([mapped.assign(is_synthetic=False), synthetic], ignore_index=True)

# Demo-only fake target -- see markdown above.
output_name = schema.output_name
weights = np.random.default_rng(2).uniform(0.1, 1.0, size=len(schema.features))
combined[output_name] = (
    combined[[f.name for f in schema.features]].to_numpy() @ weights
    + np.random.default_rng(3).normal(0, 0.1, size=len(combined))
)
combined.head()

## 5. Gate 1 -- validate the raw data before training goes anywhere near it

Two independent checks, both framework-agnostic (they run on plain values, before anything becomes a model):

1. **Physical range** -- always runs. Any value outside a feature's real-world bounds is corrupt or fabricated, full stop.
2. **PSI (Population Stability Index)** -- only runs if you pass a `reference` DataFrame of known-good prior data. Flags a dataset that's drifted too far from what's expected, even if every individual value is technically in range. We don't have a reference handy in this demo, so this part is simply skipped (not assumed to pass) -- see `report.psi_scores` coming back empty.

In [ ]:
from stamm_sdk.validation import validate_raw_data

dq_report = validate_raw_data(combined, schema)
print("passed:", dq_report.passed)
print("out_of_range:", dq_report.out_of_range)
print("psi_scores (empty -- no reference given):", dq_report.psi_scores)
print("reasons:", dq_report.reasons)

## 6. Training -- two ways

Both start from the same registry-hosted model. `FederatedClient` is the entry point for both; it needs an `APIClient` built from the same authenticated session as `sdk.ml` (you can also just use `sdk.ml` directly here -- they're the same session underneath, this is just the more explicit form).

### 6a. Mode B -- let the SDK run the training loop

Simplest path: hand it your data and hyperparameters, get back a ready-to-validate `LocalUpdate`. `hyperparams` is passed straight to the real underlying estimator's `set_params()` for sklearn models (so the valid keys depend on the specific class -- `n_estimators`/`max_depth` here because this one's a `RandomForestRegressor`; see `docs/MODEL_CATALOG.md` for every other model's real class and valid hyperparameter names). For a PyTorch model it would instead mean `epochs`/`lr`/`batch_size`, a fixed generic loop.

This simulates **site A**, one participant in a hypothetical federation.

In [ ]:
from stamm_sdk.learning import FederatedClient
from stamm_sdk.registry.api_client import APIClient

api = APIClient(session=sdk.session)
client = FederatedClient(api)

update_site_a = client.train(
    project_id, model_id, combined,
    hyperparams={"n_estimators": 50, "max_depth": 6, "min_samples_leaf": 2},
    round_number=1, participant_id="site_A",
)
print("kind:", update_site_a.kind, "| dataset_size:", update_site_a.dataset_size)

### 6b. Mode A -- you bring your own training

`load_model()` hands you the real, deserialized object -- here a fitted `RandomForestRegressor` -- and the SDK steps back entirely. Train it however you want: your own loop, a validation split, warm-starting, whatever. `extract_update()` is the only SDK call needed afterward, to package your result into the same `LocalUpdate` shape Gate 2 and aggregation expect regardless of how it was produced.

This simulates **site B**, a second, independent, honest participant with its own local data.

In [ ]:
site_b_data = generate_synthetic_rows(
    schema, pd.DataFrame(columns=[f.name for f in schema.features]), target_n=60, seed=99
)
site_b_data[output_name] = (
    site_b_data[[f.name for f in schema.features]].to_numpy() @ weights
    + np.random.default_rng(100).normal(0, 0.1, size=len(site_b_data))
)

loaded = client.load_model(project_id, model_id)
X_b, y_b = site_b_data[[f.name for f in schema.features]], site_b_data[output_name]
loaded.native_model.fit(X_b, y_b)   # <- your own training call, could be anything

update_site_b = client.extract_update(
    loaded, loaded.native_model, dataset_size=len(site_b_data),
    round_number=1, participant_id="site_B",
)
print("kind:", update_site_b.kind, "| dataset_size:", update_site_b.dataset_size)

### 6c. Training across different model types

Everything above used `0001_python_penicillin_RF`. The *calls* (`train()`, `load_model()`) look the same for any model -- but what counts as a valid `hyperparams` dict, and what actually happens inside `train()`, depends on the real underlying class:

- **sklearn models** (Random Forest, SVM, Gradient Boosting, ...): `hyperparams` is passed straight to that specific class's `set_params()` -- so the valid keys are whatever *that* class accepts, not a generic set. A `RandomForestRegressor` and an `SVR` don't share a single hyperparameter name in common.
- **Keras/PyTorch models**: `hyperparams` always means the same fixed three keys (`epochs`, `lr`, `batch_size`) -- a generic training loop, not passed to the model's constructor.

A small helper to avoid repeating the demo-data boilerplate for each one:

In [ ]:
def demo_train(model_id, hyperparams, n=60, seed=0):
    """Fetch model_id's schema, build demo data shaped for it, and train
    -- used below to compare hyperparameters across very different model
    classes without repeating the same boilerplate four times."""
    s = sdk.ml.get_schemas(project_id, model_id)
    data = generate_synthetic_rows(
        s, pd.DataFrame(columns=[f.name for f in s.features]), target_n=n, seed=seed
    )
    w = np.random.default_rng(seed + 1).uniform(0.1, 1.0, size=len(s.features))
    data[s.output_name] = (
        data[[f.name for f in s.features]].to_numpy() @ w
        + np.random.default_rng(seed + 2).normal(0, 0.1, size=len(data))
    )
    update = client.train(project_id, model_id, data, hyperparams=hyperparams, participant_id="demo")
    print(f"{model_id:35} framework={s.framework:8} kind={update.kind:8} dataset_size={update.dataset_size}")
    return update

**Random Forest** (`0001_python_penicillin_RF`) -- the one already used throughout this notebook. `RandomForestRegressor` hyperparameters: `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`, `criterion`.

`verbose` works here too (it's a real sklearn constructor param, not something this SDK adds) -- but Random Forest has no `metadata["loss_history"]` to show afterward: trees are built independently (bagging), not as a sequence that converges, so there's no equivalent of "loss per step" to capture. `verbose` only gets you progress-while-it-runs, nothing to inspect after.

In [ ]:
_ = demo_train("0001_python_penicillin_RF", {
    "n_estimators": 50, "max_depth": 6, "min_samples_leaf": 2, "verbose": 1,
})

**SVM** (`0007_python_penicillin_SVM`) -- `SVR` hyperparameters: `C`, `kernel`, `epsilon`, `gamma`. Trains fine -- but with the registry's own `kernel="rbf"`, the fitted `SVR` has no `.coef_` (only a *linear*-kernel SVR would), so `extract_update()` tags it `kind="trees"`, which is wrong for an SVM. **Don't feed this one into `FederatedAggregator.aggregate()` yet** -- it would crash looking for `.estimators_`, which `SVR` doesn't have. Training and `predict()` are fine; only the aggregation step isn't ready for this model class.

In [ ]:
_ = demo_train("0007_python_penicillin_SVM", {
    "C": 2.0, "kernel": "rbf", "epsilon": 0.01, "gamma": "scale",
})

**Gradient Boosting** (`0008_python_penicillin_GBM`) -- `GradientBoostingRegressor` hyperparameters: `n_estimators`, `max_depth`, `learning_rate`, `subsample`. This artifact was saved with scikit-learn 1.5.2 and fails to unpickle on newer versions (`ModuleNotFoundError: No module named '_loss'`) -- `stamm-sdk[learning]` pins `scikit-learn==1.5.2` exactly for this reason (see `docs/MODEL_CATALOG.md`), so this just works as long as you installed the extra normally.

Unlike Random Forest, boosting really is a sequence of stages that converges -- so `verbose=1` prints real "Iter / Train Loss / Remaining Time" progress, **and** `update.metadata["loss_history"]` comes back populated afterward (sklearn's own `train_score_`, the per-stage training deviance -- the real equivalent of the Keras loss curve above, just from a different framework).

Watch out for `0010_python_penicillin_HGB` if you try it too (same family, not shown here) -- it's `HistGradientBoostingRegressor`, which has **no `n_estimators`**; the equivalent there is `max_iter`.

In [ ]:
gbm_update = demo_train("0008_python_penicillin_GBM", {
    "n_estimators": 100, "max_depth": 4, "learning_rate": 0.05, "subsample": 0.8, "verbose": 1,
})
print("loss per stage (first 3, last 3):", gbm_update.metadata["loss_history"][:3], "...",
      gbm_update.metadata["loss_history"][-3:])

**LSTM** (`0009_python_penicillin_LSTM`) -- a real Keras/TensorFlow model, not sklearn (`algorithm="rnn"` would suggest PyTorch, but the real declared packages say Keras -- see §3). Hyperparameters here are the fixed generic set: `epochs`, `lr`, `batch_size` -- *not* passed to the model's constructor like the sklearn examples above.

`train()` reshapes your flat `(rows, features)` data into whatever 3D shape this model's own `input_shape` declares -- confirmed `(batch, 1, 8)` against the real artifact, so your data becomes `(rows, 1, 8)` automatically. Once trained, this model's `kind` comes back `"gradient"` (Keras exposes real weight tensors via `get_weights()`), so -- unlike the Random Forest used earlier -- it would go through `validate_gradient_update`'s cosine-similarity check in Gate 2, not the tree-accuracy one.

Training multiple epochs silently (no feedback at all until it's done) isn't great for actually watching how it's going, so there are two ways to see progress:
- `hyperparams={"verbose": 1}` (or `2`) passes straight to Keras's own `model.fit(verbose=...)` -- `1` gives the live progress bar you'll see below, `2` gives one line per epoch instead, `0` (the default) stays silent.
- Regardless of `verbose`, the per-epoch loss is always captured in `update.metadata["loss_history"]` -- useful for plotting a convergence curve afterward even if you didn't want console output while it ran. (PyTorch training supports the same two things, `verbose=True` and `metadata["loss_history"]`, for when there's a real PyTorch model in this registry to try it against.)

In [ ]:
lstm_update = demo_train("0009_python_penicillin_LSTM", {
    "epochs": 3, "lr": 0.001, "batch_size": 8, "verbose": 1,   # verbose=1 -> Keras's own per-epoch progress bar
})
print("weight tensors in payload:", len(lstm_update.payload))
print("loss per epoch:", lstm_update.metadata["loss_history"])

The rest of this notebook (Gate 2, aggregation, persistence) continues with the Random Forest round from §6a/6b (`update_site_a`, `update_site_b`) -- the four calls above were about comparing training mechanics across model types, not starting a new round.

## 7. A third, malicious participant -- to prove Gate 2 actually catches something

Everything so far would still "work" even if Gate 2 did nothing. To show it's a real defense and not just a rubber stamp, this trains a third model on **data whose labels have no relationship to the inputs** -- a cheap, realistic stand-in for "bad or poisoned local data." Whatever this forest learns won't generalize, which is exactly the kind of contribution a real federation needs to catch before it pollutes the global model.

This cell only prints `kind` on purpose -- it's back to the plain Random Forest from §6a/6b (not the RF/SVM/GBM/LSTM comparison from §6c), and the point here is Gate 2's defense, not training mechanics. If you want to see this attacker's own progress/`loss_history` the same way §6c did, add `"verbose": 1` to its `.fit()` call the same way -- nothing about the attacker changes which hyperparameters or metadata are available, only the data it's trained on.

In [ ]:
attacker_data = generate_synthetic_rows(
    schema, pd.DataFrame(columns=[f.name for f in schema.features]), target_n=60, seed=7
)
# Labels unrelated to the inputs -- this is what "bad/poisoned data" looks
# like in practice: whatever this forest learns won't generalize.
attacker_data[output_name] = np.random.default_rng(8).uniform(-5, 5, size=len(attacker_data))

malicious_loaded = client.load_model(project_id, model_id)
malicious_loaded.native_model.fit(
    attacker_data[[f.name for f in schema.features]], attacker_data[output_name]
)
update_attacker = client.extract_update(
    malicious_loaded, malicious_loaded.native_model, dataset_size=len(attacker_data),
    round_number=1, participant_id="attacker",
)
print("kind:", update_attacker.kind)

## 8. Gate 2 -- validate every update before aggregating

This model's `kind` is `"trees"` (Random Forest has no single weight vector to compare directions on), so the check here is performance-based: score each participant's forest against a shared reference set, then flag whoever's accuracy is a statistical outlier relative to the *other* participants in the same round -- `site_A`/`site_B` should score similarly to each other and pass; `attacker` should stand out and fail.

(For a model with `kind == "gradient"` -- a linear model or a neural net -- the equivalent check is `validate_gradient_update`, which compares update *direction* via cosine similarity instead of accuracy: an attacker can rescale a gradient's magnitude freely, but not its direction, without weakening their own attack. Same goal, different math, because the two model families don't share a comparable representation.)

In [ ]:
from stamm_sdk.validation import validate_tree_update

round_updates = [update_site_a, update_site_b, update_attacker]
reference_data = combined   # has the real feature/target relationship

for u in round_updates:
    report = validate_tree_update(u, round_updates, reference_data, output_name)
    print(f"{u.participant_id:>10}: passed={report.passed!s:5} accuracy={report.accuracy:.3f} "
          f"z={report.accuracy_vs_round} reasons={report.reasons}")

## 9. Aggregate

`FederatedAggregator.aggregate()` re-runs Gate 2 internally (you don't need to filter the list yourself first), drops whoever fails, and merges the trees of everyone who passes into one global forest -- each accepted participant contributing a `dataset_size`-proportional share of their own trees. That's the tree-ensemble equivalent of FedAvg's weighted mean: there's no meaningful way to "average" two separately-grown decision trees, so the aggregation is a merge, not a mean.

Note the attacker's rejected contribution never touches `global_payload` at all -- `aggregate()` raises instead of returning a result if *everyone* in a round fails, so a result always means at least one trusted contribution went in.

In [ ]:
from stamm_sdk.learning import FederatedAggregator

result = FederatedAggregator().aggregate(
    round_updates, history=[], reference_data=reference_data, output_name=output_name,
)
print("accepted:   ", result.accepted_participants)
print("rejected:   ", result.rejected_participants)
print("global forest n_estimators:", result.global_payload.n_estimators)
print("performance_vs_previous (R^2 on reference_data):", result.performance_vs_previous)
print("direction_ok:", result.direction_ok)

## 10. Persist the round back to the registry

A `Federation` row has to exist before you can record a round against it -- plain CRUD, same pattern as everything else in the registry. Then `run_federation_round()` runs Gate 2 + aggregation (everything from steps 8-9, in one call) and writes a `federation_rounds` row plus one `model_contributions` row per accepted participant, weighted by their share of the accepted dataset size.

It does **not** register the merged forest itself as a new `soft_sensors` catalog entry -- that's a separate decision (new version vs. overwrite) left to you. Pass that model's id in as `global_model_id` once you've registered it so the persisted rows point at it; we leave it `None` here since this is a demo round, not a real one we want to publish.

This cell is commented out by default -- uncomment it if you want to actually write a demo federation + round into the live registry.

In [ ]:
# from stamm_sdk.learning import run_federation_round
#
# federation = api.session.request("POST", "/api/v1/federations/", json={
#     "slug": "penicillin_titer_guide_demo",
#     "name": "Penicillin titer -- complete_guide.ipynb demo",
#     "aggregation_strategy": "fedavg_trees",
# })
#
# persisted = run_federation_round(
#     api, federation_id=federation["id"], round_number=1,
#     updates=round_updates, history=[],
#     reference_data=reference_data, output_name=output_name,
# )
# print(persisted)

## 11. Bonus -- `predict()` for models you can't train locally

Not every model in this registry can be `load_model()`-ed -- R models (`rpy2` territory) and Keras/TensorFlow models (not implemented yet) both raise a clear `RuntimeError` instead. They're still fully usable for **inference**: `predict()` proxies the call to the registry's own runtime for that language, server-side.

`0002_R_penicillin_RF` is an R model (`ranger` package) -- `framework` correctly comes back `"r"`, `load_model()` refuses cleanly, and `predict()` works anyway.

In [ ]:
r_model_id = "0002_R_penicillin_RF"
r_schema = sdk.ml.get_schemas(project_id, r_model_id)
print("framework:", r_schema.framework)

try:
    client.load_model(project_id, r_model_id)
except RuntimeError as e:
    print("load_model() raised as expected:", e)

# NOTE: at the time this notebook was written, this specific call returns
# a 500 ("Invalid response from R API") -- a pre-existing issue in the
# registry's own R proxy service, unrelated to this SDK. Wrapped so the
# notebook keeps going either way; if you're running this after that's
# fixed server-side, you'll see a real prediction instead of the message.
one_row = {f.name: (f.range_min + f.range_max) / 2 for f in r_schema.features}
try:
    prediction = sdk.ml.predict(project_id, r_model_id, one_row)
    print("predict() result:", prediction)
except Exception as e:
    print(f"predict() failed ({type(e).__name__}) -- registry-side R proxy issue, not an SDK bug: {e}")

## 12. Where to go next

You've now touched every stage: discovery, schema inspection, data mapping, synthetic data, Gate 1, both training modes, Gate 2 (with a real adversarial case), aggregation, and persistence -- plus the `predict()` fallback for models you can't train locally.

- `docs/SDK_GUIDE.md` -- full reference, every module and import in one table.
- `docs/MODEL_CATALOG.md` -- every other model in this registry: which ones work, which need a pinned sklearn version, which are mislabeled, which aren't supported yet, and why.
- `notebooks/model_catalog_demo.ipynb` -- the hyperparameter cheat-sheet for each specific model class.
- `notebooks/federated_learning_demo.ipynb` -- the original, shorter version of this walkthrough.

## Cleanup

In [ ]:
sdk.logout()
sdk.close()
print("Done")